### SoRL Playground

This notebook demonstrates the SoRL post-training pipeline. Porting a OSS model, and adopt SoRL trainer to post-train the model accordingly. 

In [1]:
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

# Disable MPS for stability
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import TrainingArguments, AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper
from sorl.sorl_trainer import SorlTrainer

device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
# Initialize SoRL model
from sorl.sorl_wrapper import SorlModelWrapper
model_name = "Qwen/Qwen2.5-0.5B"
model = SorlModelWrapper.from_pretrained(
    model_name,
    abstract_vocab_size_list=[128],
)
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name) 

Some weights of Qwen2ForCausalLM were not initialized from the model checkpoint at Qwen/Qwen2.5-0.5B and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [6]:
# We might be missing the "mask" gadget in the trainer, if we want to 
# use GSM8K alike dataset --- is there a build-in approach for it? 


# Replace your SimpleDataset with GSM8K data loading
from datasets import load_dataset
 
# Load GSM8K dataset
gsm8k_dataset = load_dataset("gsm8k", "main", split="train")
 
# Create dataset class for GSM8K
class GSM8KDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, tokenizer, max_length=16):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        example = self.dataset[idx]
        
        # Format for math reasoning
        text = f"Question: {example['question']}\nAnswer: {example['answer']}"
        
        # Tokenize
        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoded["input_ids"].squeeze(),
            "attention_mask": encoded["attention_mask"].squeeze()
        }
 
# Use GSM8K dataset
train_dataset = GSM8KDataset(gsm8k_dataset, tokenizer)

# Training arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./sorl_results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    warmup_steps=1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_steps=10,
    eval_steps=10,
)

# Create SoRL trainer
print("Creating SoRL trainer...")
trainer = SorlTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    num_rollouts=4,
    K=3,
    max_iterations=2,
    memory_span_abs=1792,
    memory_span_traj=1792,
    temperature=1.0,
    alpha_info_gain=10.0,
    alpha_abs=0.1,
    alpha_soft_zipf=1.0,
)

print("Trainer created successfully!")
print(f"Model vocab sizes: {model.vocab_sizes}")
print(f"Total vocab size: {model.total_vocab_size}")

Creating SoRL trainer...
Trainer created successfully!
Model vocab sizes: tensor([151936,    129])
Total vocab size: 152065


/Users/ksgk/Implementation/mod_gpt/sorl/sorl_trainer.py:222: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SorlTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


In [ ]:
# ============================================================
# Dataset & Accuracy Evaluation (from data/pt_dataset.py)
# ============================================================
from data.pt_dataset import get_dataset, evaluate_accuracy, collate_fn

# Quick test
dataset_name = "gsm8k"
train_dataset = get_dataset(dataset_name, split="train", tokenizer=tokenizer, max_length=128)
print(f"Dataset: {dataset_name}, size: {len(train_dataset)}")
print(f"Sample keys: {list(train_dataset[0].keys())}")
print(f"Sample shape: {train_dataset[0]['input_ids'].shape}")

Dataset: gsm8k, size: 7473
Sample keys: ['input_ids', 'attention_mask']
Sample shape: torch.Size([128])


In [ ]:
# ============================================================
# SoRL Training with standalone Trainer
# ============================================================
from sorl.trainer import SoRLTrainer, SoRLConfig
from data.pt_dataset import get_dataset, evaluate_accuracy

config = SoRLConfig(
    # SoRL search
    num_rollouts=4, K=4, max_iterations=2,
    memory_span_abs=1792, memory_span_traj=1792, temperature=1.0,
    # Loss weights
    alpha_info_gain=10.0, alpha_abs=0.1, alpha_soft_zipf=1.0,
    # Optimizer
    lr=1e-5, weight_decay=0.01, warmup_steps=50, max_grad_norm=1.0,
    # Training
    batch_size=2, num_epochs=3,
    # Logging / Eval / Checkpoint
    log_every=10, eval_every=500, save_every=500, eval_samples=50,
    output_dir="./ckpt/sorl",
)

# Datasets
train_ds = get_dataset("gsm8k", split="train", tokenizer=tokenizer, max_length=16)
val_ds = get_dataset("gsm8k", split="test", tokenizer=tokenizer, max_length=16)

trainer = SoRLTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    val_dataset=val_ids,  # set val_ds here for periodic eval
    compute_accuracy=evaluate_accuracy,
    config=config,
    ddp=False,  # set True when using torchrun
)

history = trainer.train()

Total steps: 11211 | Steps/epoch: 3737 | Effective batch: 2
E1 S10/11211 | loss=24.9728 base=22.8887 info=-0.5750 abs=23.5640 zipf=5.4776 | lr=1.80e-06 | 18s 
E1 S20/11211 | loss=6.3436 base=18.5485 info=-1.9547 abs=20.5770 zipf=5.2847 | lr=3.80e-06 | 36s 
E1 S30/11211 | loss=-10.0185 base=18.0427 info=-3.4871 abs=19.2061 zipf=4.8890 | lr=5.80e-06 | 54s 
E1 S40/11211 | loss=-44.9889 base=18.1311 info=-6.9365 abs=16.9256 zipf=4.5522 | lr=7.80e-06 | 72s 
E1 S50/11211 | loss=-19.9686 base=15.6052 info=-4.1282 abs=15.5522 zipf=4.1534 | lr=9.80e-06 | 91s 
E1 S60/11211 | loss=-1.0307 base=10.9524 info=-1.7152 abs=14.8311 zipf=3.6856 | lr=1.00e-05 | 110s 
E1 S70/11211 | loss=-9.1235 base=12.8035 info=-2.6501 abs=13.9547 zipf=3.1781 | lr=1.00e-05 | 129s 
E1 S80/11211 | loss=5.3116 base=9.8526 info=-0.8637 abs=13.5058 zipf=2.7454 | lr=1.00e-05 | 148s 
